# Álgebra Linear

  Buscando desenvolver a base para álgebra linear, foram importadas constantes e funções de outros módulos programados anteriormente. 
  
  Também foi definida as classes "Matriz", essa é uma representação das matrizes. Nela, foram definidas as partes que compõem uma matriz na classe, sendo suas linhas e colunas. Também foram definidas as maneiras com que a matriz é interpretada internamente, como ela aparecerá ao ser printada, como aparecerá internamente no código, dentre outras propriedades. Com o mesmo objetivo de implementar matrizes, foram definidas como as operações de adição, subtração, multiplicação e a negação de matrizes funciona.Em seguida, foram desenvolvidas funções para conferir propriedades específicas de certas matrizes, sendo a verificação se é uma matriz quadrada, se é uma matriz simétrica e se duas matrizes possuem as mesmas dimensões, além de propriedades estruturais de se é diagonal, trângular superior, triângular inferior e se é ortogonal.

  Para as operações de algebra linear, foram feitos códigos para realizar adição e subtração de matrizes, multiplicação de matrizes, multiplicação de matriz por escalar e potenciação de matrizes. Em sequência, foram feitos algoritmos para a encontrar a matriz transposta, submatriz, determinante, cofatores, matriz de cofatores, adjunta, inversa e o posto.

  Para resolver sistemas lineares, Foram desenvolvidos algoritmos para realizar a regra de cramer, eliminação gausseana, método de Gauss-Jordan, decomposição LU, decomposição Cholesky, decomposição QR, método de Jacobi, método de Gauss-Seidel e método dos mínimos quadrados. Também há algoritmos de construção de matrizes especiais, esses são matriz identidade, nula, de uns e de vetores linha.

  Após isso, foi definida as classes "Vetor", essa é uma representação de vetores. Nela, foram definidas as partes que compõem um vetor na classe. Também foram definidas as maneiras com que o vetor é interpretado internamente, como ele aparecerá ao ser printada, como aparecerá internamente no código, dentre outras propriedades, além de como as operações do python são interpretadas pela classe.

  Em seguida foi definido para os vetores as operações de adição e subtração de vetores, multiplicação por escalar, produto interno, norma e normalização. Também foram definidos algoritmos que encontram a distância euclidiana, o ângulo entre dois vetores, que verificam se dois vetores são ortogonais, a projeção ortogonal de um vetor por outro, produto vetorial, conversões entre matriz coluna e vetor, autovalores e autovetores, traço, e norma pela norma de Frobenius e por produto de Hadamard.

  Para análises, foram programados códigos para aplicar transformações, verificar se uma transformação é linear, comparar vetores, encontrar a matriz associada a uma transformação linear. E, por fim, foram feitos um para base para núcleo e para dimensão da imagem.

  Para finalizar, adicionou-se a linha de código "%%writefile algebra_linear.py" para converter o código em um módulo exportável para ser usado em outros módulos e no uso final. Pelo mesmo motivo, o código teve que ser colocado inteiramente em uma única célula de código, pois o comando usado para isso exporta códigos de apenas uma célula para isso.

In [1]:
%%writefile algebra_linear.py

from constantes import tolerancia_de_erro, maximo_de_interacoes
from aritmetica import valor_absoluto, potencia_rapida_int

#Definição de classe para matrizes
class Matriz:
    
    __slots__ = ("_dados", "_linhas", "_colunas")

    def __init__(proprio, dados: list):
        if not dados or not dados[0]:
            raise ValueError("Matriz não pode ser vazia.")
        n_colunas = len(dados[0])
        for i, linha in enumerate(dados):
            if len(linha) != n_colunas:
                raise ValueError(f"Linha {i} tem {len(linha)} colunas, esperado {n_colunas}.")
        proprio._dados = [[float(v) for v in linha] for linha in dados]
        proprio._linhas = len(dados)
        proprio._colunas = n_colunas
        

    @property
    def linhas(proprio) -> int:
        return proprio._linhas
        

    @property
    def colunas(proprio) -> int:
        return proprio._colunas
        

    @property
    def forma(proprio) -> tuple:
        return (proprio._linhas, proprio._colunas)
        

    def __getitem__(proprio, indice):
        return proprio._dados[indice]
        

    def __str__(proprio) -> str:
        linhas_formatadas = []
        for linha in proprio._dados:
            formatado = [f"{v:10.4f}" for v in linha]
            linhas_formatadas.append("[ " + "  ".join(formatado) + " ]")
        return "\n".join(linhas_formatadas)
        

    def __repr__(proprio) -> str:
        return f"Matriz({proprio._dados})"
        

    def __eq__(proprio, outro) -> bool:
        if not isinstance(outro, Matriz):
            return False
        if proprio.forma != outro.forma:
            return False
        for i in range(proprio._linhas):
            for j in range(proprio._colunas):
                if valor_absoluto(proprio._dados[i][j] - outro._dados[i][j]) > tolerancia_de_erro:
                    return False
        return True
        

    # Interpretação das operações do Python
    def __add__(proprio, outro: "Matriz") -> "Matriz":
        _verificar_mesma_forma(proprio, outro, "soma")
        resultado = []
        for i in range(proprio._linhas):
            linha = [proprio._dados[i][j] + outro._dados[i][j] for j in range(proprio._colunas)]
            resultado.append(linha)
        return Matriz(resultado)
        

    def __sub__(proprio, outro: "Matriz") -> "Matriz":
        _verificar_mesma_forma(proprio, outro, "subtração")
        resultado = []
        for i in range(proprio._linhas):
            linha = [proprio._dados[i][j] - outro._dados[i][j] for j in range(proprio._colunas)]
            resultado.append(linha)
        return Matriz(resultado)
        

    def __mul__(proprio, outro) -> "Matriz":
        if isinstance(outro, (int, float)):
            return Matriz([[proprio._dados[i][j] * outro for j in range(proprio._colunas)]
                           for i in range(proprio._linhas)])
        if isinstance(outro, Matriz):
            if proprio._colunas != outro._linhas:
                raise ValueError(f"Incompatível: {proprio.forma} x {outro.forma} — " f"colunas de A ({proprio._colunas}) != linhas de B ({outro._linhas}).")
            resultado = [[0.0] * outro._colunas for _ in range(proprio._linhas)]
            for i in range(proprio._linhas):
                for j in range(outro._colunas):
                    s = 0.0
                    for k in range(proprio._colunas):
                        s += proprio._dados[i][k] * outro._dados[k][j]
                    resultado[i][j] = s
            return Matriz(resultado)
        return NotImplemented
        

    def __rmul__(proprio, outro) -> "Matriz":
        if isinstance(outro, (int, float)):
            return proprio.__mul__(outro)
        return NotImplemented
        

    def __neg__(proprio) -> "Matriz":
        return proprio.__mul__(-1)


#Verificação de propriedades específicas
#Verificação se é uma matriz quadrada
def eh_quadrada(A: Matriz) -> bool:
    return A._linhas == A._colunas


#Verificação se é uma matriz simétrica
def eh_simetrica(A: Matriz) -> bool:
    if not eh_quadrada(A):
        return False
    for i in range(A._linhas):
        for j in range(i + 1, A._colunas):
            if valor_absoluto(A._dados[i][j] - A._dados[j][i]) > tolerancia_de_erro:
                return False
    return True


#Verificações de propriedades estruturais
#Verificação se é diagonal
def eh_diagonal(A: Matriz) -> bool:
    if not eh_quadrada(A):
        return False
    for i in range(A._linhas):
        for j in range(A._colunas):
            if i != j and valor_absoluto(A[i][j]) > tolerancia_de_erro:
                return False
    return True


#Verificação se é triangular superior
def eh_triangular_superior(A: Matriz) -> bool:
    if not eh_quadrada(A):
        return False
    for i in range(A._linhas):
        for j in range(i):
            if valor_absoluto(A[i][j]) > tolerancia_de_erro:
                return False
    return True


#Verificação se é triangular inferior
def eh_triangular_inferior(A: Matriz) -> bool:
    if not eh_quadrada(A):
        return False
    for i in range(A._linhas):
        for j in range(i + 1, A._colunas):
            if valor_absoluto(A[i][j]) > tolerancia_de_erro:
                return False
    return True


#Verificação se é ortogonal
def eh_ortogonal(A: Matriz) -> bool:
    if not eh_quadrada(A):
        return False
    produto = multiplicacao_matrizes(A, transposta(A))
    I = matriz_identidade(A._linhas)
    return produto == I


#Acessos utilitários
def obter(A: Matriz, i: int, j: int) -> float:
    return A._dados[i][j]


def definir(A: Matriz, i: int, j: int, valor: float):
    A._dados[i][j] = float(valor)


def copiar(A: Matriz) -> Matriz:
    return Matriz([linha[:] for linha in A._dados])


#Operações Matriciais
#Adição de matrizes
def adicao_matrizes(A: Matriz, B: Matriz) -> Matriz:
    return A + B


#Subtração de matrizes
def subtracao_matrizes(A: Matriz, B: Matriz) -> Matriz:
    return A - B


#Multiplicação de matrizes
def multiplicacao_matrizes(A: Matriz, B: Matriz) -> Matriz:
    return A * B


#Multiplicação de matriz por escalar
def multiplicacao_matriz_escalar(escalar: float, A: Matriz) -> Matriz:
    return A * escalar


#Negação de matriz
def negacao_matriz(A: Matriz) -> Matriz:
    return -A


#Potenciação de matrizes
def potencia_matriz(A: Matriz, n: int) -> Matriz:
    if not eh_quadrada(A):
        raise ValueError("Potência de matriz requer matriz quadrada.")
    if n < 0:
        raise ValueError("Expoente negativo requer inversa; use 'potencia_matriz(inversa(A), -n)'.")
    if n == 0:
        return matriz_identidade(A._linhas)
    if n == 1:
        return copiar(A)
    if n % 2 == 0:
        metade = potencia_matriz(A, n // 2)
        return multiplicacao_matrizes(metade, metade)
    return multiplicacao_matrizes(A, potencia_matriz(A, n - 1))


#Matriz transposta
def transposta(A: Matriz) -> Matriz:
    resultado = [[A._dados[i][j] for i in range(A._linhas)]
                 for j in range(A._colunas)]
    return Matriz(resultado)


#Submatriz
def submatriz(A: Matriz, pular_linha: int, pular_coluna: int) -> Matriz:
    resultado = []
    for i in range(A._linhas):
        if i == pular_linha:
            continue
        linha = [A._dados[i][j] for j in range(A._colunas) if j != pular_coluna]
        resultado.append(linha)
    return Matriz(resultado)


#Determinante
def determinante(A: Matriz) -> float:
    if not eh_quadrada(A):
        raise ValueError("Determinante requer matriz quadrada.")
    n = A._linhas
    if n == 1:
        return A._dados[0][0]
    if n == 2:
        return A._dados[0][0] * A._dados[1][1] - A._dados[0][1] * A._dados[1][0]

    M = [linha[:] for linha in A._dados]
    sinal_determinante = 1
    valor_determinante = 1.0

    for coluna in range(n):
        linha_pivo = coluna
        maior_valor = valor_absoluto(M[coluna][coluna])
        for l in range(coluna + 1, n):
            if valor_absoluto(M[l][coluna]) > maior_valor:
                maior_valor = valor_absoluto(M[l][coluna])
                linha_pivo = l

        if valor_absoluto(M[linha_pivo][coluna]) < tolerancia_de_erro:
            return 0.0

        if linha_pivo != coluna:
            M[coluna], M[linha_pivo] = M[linha_pivo], M[coluna]
            sinal_determinante *= -1

        valor_determinante *= M[coluna][coluna]

        for l in range(coluna + 1, n):
            if valor_absoluto(M[l][coluna]) < tolerancia_de_erro:
                continue
            fator = M[l][coluna] / M[coluna][coluna]
            for c in range(coluna, n):
                M[l][c] -= fator * M[coluna][c]

    return sinal_determinante * valor_determinante


#Cofator
def cofator(A: Matriz, i: int, j: int) -> float:
    sinal_cofator = 1 if (i + j) % 2 == 0 else -1
    return sinal_cofator * determinante(submatriz(A, i, j))


#Matriz de cofatores
def matriz_de_cofatores(A: Matriz) -> Matriz:
    if not eh_quadrada(A):
        raise ValueError("Matriz de cofatores requer matriz quadrada.")
    n = A._linhas
    resultado = [[cofator(A, i, j) for j in range(n)] for i in range(n)]
    return Matriz(resultado)


#Matriz adjunta
def adjunta(A: Matriz) -> Matriz:
    return transposta(matriz_de_cofatores(A))


#Matriz inversa
def inversa(A: Matriz) -> Matriz:
    if not eh_quadrada(A):
        raise ValueError("Inversa requer matriz quadrada.")
        
    n = A._linhas

    aumentada = [A._dados[i][:] + [1.0 if i == j else 0.0 for j in range(n)]
                 for i in range(n)]

    for coluna in range(n):
        pivo = coluna
        maior_valor = valor_absoluto(aumentada[coluna][coluna])
        for l in range(coluna + 1, n):
            if valor_absoluto(aumentada[l][coluna]) > maior_valor:
                maior_valor = valor_absoluto(aumentada[l][coluna])
                pivo = l

        if valor_absoluto(aumentada[pivo][coluna]) < tolerancia_de_erro:
            raise ValueError("Matriz singular, com determinante ≈ 0, não possui inversa.")

        aumentada[coluna], aumentada[pivo] = aumentada[pivo], aumentada[coluna]

        valor_pivo = aumentada[coluna][coluna]
        aumentada[coluna] = [v / valor_pivo for v in aumentada[coluna]]

        for l in range(n):
            if l == coluna:
                continue
            fator = aumentada[l][coluna]
            aumentada[l] = [aumentada[l][c] - fator * aumentada[coluna][c] for c in range(2 * n)]

    dados_inversa = [aumentada[i][n:] for i in range(n)]
    return Matriz(dados_inversa)


#Posto
def posto(A: Matriz) -> int:
    M = [linha[:] for linha in A._dados]
    m, n = A._linhas, A._colunas
    l = 0

    for coluna in range(n):
        if l >= m:
            break
        pivo = -1
        for linha in range(l, m):
            if valor_absoluto(M[linha][coluna]) > tolerancia_de_erro:
                pivo = linha
                break
        if pivo == -1:
            continue

        M[l], M[pivo] = M[pivo], M[l]
        valor_pivo = M[l][coluna]
        M[l] = [v / valor_pivo for v in M[l]]

        for linha in range(m):
            if linha == l or valor_absoluto(M[linha][coluna]) < tolerancia_de_erro:
                continue
            f = M[linha][coluna]
            M[linha] = [M[linha][c] - f * M[l][c] for c in range(n)]
        l += 1

    return l

#Resolução de sistemas lineares
#Regra de Cramer
def cramer(A: list, b: list) -> list:
    n = len(A)
    det_A = determinante(Matriz(A))

    if valor_absoluto(det_A) < tolerancia_de_erro:
        raise ValueError("Sistema singular: det(A) ≈ 0, sem solução única por Cramer.")

    solucao = []
    for i in range(n):
        A_i = [linha[:] for linha in A]
        for linha in range(n):
            A_i[linha][i] = b[linha]
        det_A_i = determinante(Matriz(A_i))
        solucao.append(det_A_i / det_A)

    return solucao


#Eliminação gaussiana
def eliminacao_gaussiana(A: Matriz) -> Matriz:
    M = [linha[:] for linha in A._dados]
    m, n = A._linhas, A._colunas
    l = 0

    for coluna in range(n):
        if l >= m:
            break
        pivo = -1
        maior_valor = 0.0
        for linha in range(l, m):
            if valor_absoluto(M[linha][coluna]) > maior_valor:
                maior_valor = valor_absoluto(M[linha][coluna])
                pivo = linha
        if pivo == -1 or maior_valor < tolerancia_de_erro:
            continue
        M[l], M[pivo] = M[pivo], M[l]
        for linha in range(l + 1, m):
            if valor_absoluto(M[linha][coluna]) < tolerancia_de_erro:
                continue
            fator = M[linha][coluna] / M[l][coluna]
            M[linha] = [M[linha][c] - fator * M[l][c] for c in range(n)]
        l += 1

    return Matriz(M)


#Método de Gauss-Jordan
def gauss_jordan(A: Matriz) -> Matriz:
    M = [linha[:] for linha in A._dados]
    m, n = A._linhas, A._colunas
    l = 0

    for coluna in range(n):
        if l >= m:
            break
        pivo = -1
        maior_valor = 0.0
        for linha in range(l, m):
            if valor_absoluto(M[linha][coluna]) > maior_valor:
                maior_valor = valor_absoluto(M[linha][coluna])
                pivo = linha
        if pivo == -1 or maior_valor < tolerancia_de_erro:
            continue
        M[l], M[pivo] = M[pivo], M[l]
        valor_pivo = M[l][coluna]
        M[l] = [v / valor_pivo for v in M[l]]
        for linha in range(m):
            if linha == l or valor_absoluto(M[linha][coluna]) < tolerancia_de_erro:
                continue
            f = M[linha][coluna]
            M[linha] = [M[linha][c] - f * M[l][c] for c in range(n)]
        l += 1

    return Matriz(M)

#Decomposição LU
def decomposicao_lu(A: Matriz) -> tuple:
    if not eh_quadrada(A):
        raise ValueError("Decomposição LU requer matriz quadrada.")
    n = A._linhas

    U = [linha[:] for linha in A._dados]
    L = [[1.0 if i == j else 0.0 for j in range(n)] for i in range(n)]
    permutacao = list(range(n))

    for k in range(n):
        maior_valor = valor_absoluto(U[k][k])
        pivo = k
        for l in range(k + 1, n):
            if valor_absoluto(U[l][k]) > maior_valor:
                maior_valor = valor_absoluto(U[l][k])
                pivo = l

        if maior_valor < tolerancia_de_erro:
            raise ValueError("LU requer matriz não-singular.")

        if pivo != k:
            U[k], U[pivo] = U[pivo], U[k]
            permutacao[k], permutacao[pivo] = permutacao[pivo], permutacao[k]
            for c in range(k):
                L[k][c], L[pivo][c] = L[pivo][c], L[k][c]

        for l in range(k + 1, n):
            if valor_absoluto(U[k][k]) < tolerancia_de_erro:
                raise ValueError("Matriz singular durante LU. LU requer matriz não-singular.")
            L[l][k] = U[l][k] / U[k][k]
            for c in range(k, n):
                U[l][c] -= L[l][k] * U[k][c]

    dados_P = [[1.0 if permutacao[i] == j else 0.0 for j in range(n)] for i in range(n)]
    return Matriz(L), Matriz(U), Matriz(dados_P)


#Decomposição de Cholesky
def decomposicao_cholesky(A: Matriz) -> Matriz:
    
    from funcoes import sqrt

    if not eh_quadrada(A):
        raise ValueError("Cholesky requer matriz quadrada.")
    if not eh_simetrica(A):
        raise ValueError("Cholesky requer matriz simétrica.")

    n = A._linhas
    L = [[0.0] * n for _ in range(n)]

    for i in range(n):
        for j in range(i + 1):
            soma = 0.0
            for k in range(j):
                soma += L[i][k] * L[j][k]

            if i == j:
                valor_diagonal = A[i][i] - soma
                if valor_diagonal <= tolerancia_de_erro:
                    raise ValueError("Matriz que não é positiva-definida possui Cholesky indefinido.")
                L[i][j] = sqrt(valor_diagonal)
            else:
                L[i][j] = (A[i][j] - soma) / L[j][j]

    return Matriz(L)


#Decomposição QR
def decomposicao_qr(A: Matriz) -> tuple:
    m, n = A._linhas, A._colunas
    colunas_a = [Vetor([A[i][j] for i in range(m)]) for j in range(n)]
    colunas_q = []

    for j in range(n):
        u = colunas_a[j]
        for q in colunas_q:
            u = u - projecao(colunas_a[j], q)
        norma_u = norma(u)
        if norma_u < tolerancia_de_erro:
            raise ValueError("QR é indefinido, pois as colunas são linearmente dependentes.")
        colunas_q.append(u * (1.0 / norma_u))

    dados_Q = [[colunas_q[j][i] for j in range(n)] for i in range(m)]
    Q = Matriz(dados_Q)

    dados_R = [[0.0] * n for _ in range(n)]
    for i in range(n):
        for j in range(i, n):
            dados_R[i][j] = produto_interno(colunas_q[i], colunas_a[j])
    R = Matriz(dados_R)

    return Q, R


#Solução do sistema
def resolver_sistema(A: Matriz, b: list) -> list:
    if not eh_quadrada(A):
        raise ValueError("Resolver sistema requer matriz quadrada.")
    n = A._linhas
    if len(b) != n:
        raise ValueError(f"Vetor deve ter {n} elementos.")

    L, U, P = decomposicao_lu(A)

    c = [0.0] * n
    for i in range(n):
        for j in range(n):
            c[i] += P[i][j] * b[j]

    y = [0.0] * n
    for i in range(n):
        s = c[i]
        for j in range(i):
            s -= L[i][j] * y[j]
        y[i] = s

    x = [0.0] * n
    for i in range(n - 1, -1, -1):
        if valor_absoluto(U[i][i]) < tolerancia_de_erro:
            raise ValueError("Sistema singular (sem solução única).")
        s = y[i]
        for j in range(i + 1, n):
            s -= U[i][j] * x[j]
        x[i] = s / U[i][i]

    return x


#Método de Jacobi
def jacobi(A: Matriz, b: list, interacoes: int = None) -> list:
    if not eh_quadrada(A):
        raise ValueError("Jacobi requer matriz quadrada.")
    n = A._linhas
    if interacoes is None:
        interacoes = maximo_de_interacoes

    for i in range(n):
        if valor_absoluto(A[i][i]) < tolerancia_de_erro:
            raise ValueError(f"Jacobi indefinido, pois o elemento diagonal A[{i}][{i}] ≈ 0.")

    x = [0.0] * n
    for _ in range(interacoes):
        x_novo = [0.0] * n
        for i in range(n):
            soma = 0.0
            for j in range(n):
                if j != i:
                    soma += A[i][j] * x[j]
            x_novo[i] = (b[i] - soma) / A[i][i]

        diferenca = max(valor_absoluto(x_novo[i] - x[i]) for i in range(n))
        x = x_novo
        if diferenca < tolerancia_de_erro:
            break

    return x


#Método de Gauss-Seidel
def gauss_seidel(A: Matriz, b: list, interacoes: int = None) -> list:
    if not eh_quadrada(A):
        raise ValueError("Gauss-Seidel requer matriz quadrada.")
    n = A._linhas
    if interacoes is None:
        interacoes = maximo_de_interacoes

    for i in range(n):
        if valor_absoluto(A[i][i]) < tolerancia_de_erro:
            raise ValueError(f"Gauss-Seidel indefinido, pois o elemento diagonal A[{i}][{i}] ≈ 0.")

    x = [0.0] * n
    for _ in range(interacoes):
        x_anterior = x[:]
        for i in range(n):
            soma = 0.0
            for j in range(n):
                if j != i:
                    soma += A[i][j] * x[j]
            x[i] = (b[i] - soma) / A[i][i]

        diferenca = max(valor_absoluto(x[i] - x_anterior[i]) for i in range(n))
        if diferenca < tolerancia_de_erro:
            break

    return x


#Método dos mínimos quadrados
def minimos_quadrados(A: Matriz, b: list) -> list:
    At = transposta(A)
    AtA = multiplicacao_matrizes(At, A)

    n = A._colunas
    Atb = [0.0] * n
    for i in range(n):
        soma = 0.0
        for k in range(A._linhas):
            soma += At[i][k] * b[k]
        Atb[i] = soma

    return resolver_sistema(AtA, Atb)

    
#Construtores de matrizes especiais
#Identidade
def matriz_identidade(n: int) -> Matriz:
    return Matriz([[1.0 if i == j else 0.0 for j in range(n)] for i in range(n)])


#Matriz nula
def matriz_nula(m: int, n: int) -> Matriz:
    return Matriz([[0.0] * n for _ in range(m)])


#Matriz de uns
def matriz_uns(m: int, n: int) -> Matriz:
    return Matriz([[1.0] * n for _ in range(m)])


#Matriz de vetores linha
def matriz_vetores_linha(linhas: list) -> Matriz:
    return Matriz(linhas)


#Verificação se duas matrizes possuem mesma dimensão
def verificar_mesma_forma(A: Matriz, B: Matriz, operacao: str):
    if A.forma != B.forma:
        raise ValueError(f"Incompatível para {operacao}: {A.forma} != {B.forma}.")



#Definição de classe para vetores
class Vetor:

    __slots__ = ("_dados", "_dimensao")

    def __init__(proprio, dados: list):
        if not dados:
            raise ValueError("Vetor não pode ser vazio.")
        proprio._dados = [float(v) for v in dados]
        proprio._dimensao = len(dados)

    @property
    def dimensao(proprio) -> int:
        return proprio._dimensao

    def __getitem__(proprio, indice):
        return proprio._dados[indice]

    def __str__(proprio) -> str:
        formatado = [f"{v:.4f}" for v in proprio._dados]
        return "(" + ", ".join(formatado) + ")"

    def __repr__(proprio) -> str:
        return f"Vetor({proprio._dados})"

    def __eq__(proprio, outro) -> bool:
        if not isinstance(outro, Vetor) or proprio._dimensao != outro._dimensao:
            return False
        for i in range(proprio._dimensao):
            if valor_absoluto(proprio._dados[i] - outro._dados[i]) > tolerancia_de_erro:
                return False
        return True

        
    #Interpretação das operações do Python
    def __add__(proprio, outro: "Vetor") -> "Vetor":
        verificar_mesma_dimensao(proprio, outro, "soma")
        return Vetor([proprio._dados[i] + outro._dados[i] for i in range(proprio._dimensao)])

    def __sub__(proprio, outro: "Vetor") -> "Vetor":
        verificar_mesma_dimensao(proprio, outro, "subtração")
        return Vetor([proprio._dados[i] - outro._dados[i] for i in range(proprio._dimensao)])

    def __mul__(proprio, escalar) -> "Vetor":
        if isinstance(escalar, (int, float)):
            return Vetor([v * escalar for v in proprio._dados])
        return NotImplemented

    def __rmul__(proprio, escalar) -> "Vetor":
        return proprio.__mul__(escalar)

    def __neg__(proprio) -> "Vetor":
        return proprio.__mul__(-1)


#Verificação se dois vetores possuem a mesma dimensão
def verificar_mesma_dimensao(u: Vetor, v: Vetor, operacao: str):
    if u.dimensao != v.dimensao:
        raise ValueError(f"Incompatível para {operacao}: dimensão {u.dimensao} != {v.dimensao}.")


#Operações Vetoriais
#Adição
def adicao_vetores(u: Vetor, v: Vetor) -> Vetor:
    return u + v


#Subtração
def subtracao_vetores(u: Vetor, v: Vetor) -> Vetor:
    return u - v


#Multiplicação por escalar
def multiplicacao_vetor_escalar(escalar: float, v: Vetor) -> Vetor:
    return v * escalar


#Produto interno
def produto_interno(u: Vetor, v: Vetor) -> float:
    verificar_mesma_dimensao(u, v, "produto interno")
    soma = 0.0
    for i in range(u.dimensao):
        soma += u[i] * v[i]
    return soma


#Norma
def norma(v: Vetor) -> float:
    
    from funcoes import sqrt
    
    return sqrt(produto_interno(v, v))


#Normalização
def normalizar(v: Vetor) -> Vetor:
    n = norma(v)
    if n < tolerancia_de_erro:
        raise ZeroDivisionError("Não é possível normalizar o vetor nulo.")
    return v * (1.0 / n)


#Distância euclidiana
def distancia(u: Vetor, v: Vetor) -> float:
    return norma(u - v)


#Ângulo entre dois vetores
def angulo_entre(u: Vetor, v: Vetor) -> float:
    
    from trigonometria import arccos

    if norma(u) < tolerancia_de_erro or norma(v) < tolerancia_de_erro:
        raise ZeroDivisionError("Ângulo indefinido para o vetor nulo.")
    cos_theta = produto_interno(u, v) / (norma(u) * norma(v))
    if cos_theta > 1.0:
        cos_theta = 1.0
    if cos_theta < -1.0:
        cos_theta = -1.0
    return arccos(cos_theta)


#Verificação se dois vetores são ortogonais entre si
def sao_ortogonais(u: Vetor, v: Vetor) -> bool:
    return valor_absoluto(produto_interno(u, v)) < tolerancia_de_erro


#Projeção ortogonal de um vetor sobre outro
def projecao(u: Vetor, v: Vetor) -> Vetor:
    denominador = produto_interno(v, v)
    if denominador < tolerancia_de_erro:
        raise ZeroDivisionError("Não é possível projetar sobre o vetor nulo.")
    fator = produto_interno(u, v) / denominador
    return v * fator


#Produto vetorial
def produto_vetorial(u: Vetor, v: Vetor) -> Vetor:
    if u.dimensao != 3 or v.dimensao != 3:
        raise ValueError("Produto vetorial requer vetores em R^3.")
    return Vetor([
        u[1] * v[2] - u[2] * v[1],
        u[2] * v[0] - u[0] * v[2],
        u[0] * v[1] - u[1] * v[0],
    ])


#Conversão de vetor para matriz coluna
def vetor_para_matriz_coluna(v: Vetor) -> Matriz:
    return Matriz([[v[i]] for i in range(v.dimensao)])


#Conversão de matriz coluna para vetor
def matriz_coluna_para_vetor(A: Matriz) -> Vetor:
    if A.colunas != 1:
        raise ValueError(f"Esperada matriz coluna (n×1), recebido {A.forma}.")
    return Vetor([A[i][0] for i in range(A.linhas)])


#Autovalores e autovetores
#Método das Potências
def metodo_das_potencias(A: Matriz, interacoes: int = None) -> tuple:
    if not eh_quadrada(A):
        raise ValueError("Método das potências requer matriz quadrada.")

    n = A.linhas
    if interacoes is None:
        interacoes = maximo_de_interacoes

    v = Vetor([1.0] * n)
    v = normalizar(v)

    autovalor_anterior = 0.0
    for _ in range(interacoes):
        Av = aplicar_transformacao(A, v)
        norma_Av = norma(Av)
        if norma_Av < tolerancia_de_erro:
            raise ValueError("Matriz singular na direção do vetor inicial.")
        v = Av * (1.0 / norma_Av)

        Av = aplicar_transformacao(A, v)
        autovalor = produto_interno(v, Av)

        if valor_absoluto(autovalor - autovalor_anterior) < tolerancia_de_erro:
            break
        autovalor_anterior = autovalor

    return autovalor, v


#Traço
def traco(A: Matriz) -> float:
    if not eh_quadrada(A):
        raise ValueError("Traço requer matriz quadrada.")
    soma = 0.0
    for i in range(A._linhas):
        soma += A[i][i]
    return soma


#Normas
#Norma de Frobenius
def norma_frobenius(A: Matriz) -> float:
    
    from funcoes import sqrt
    
    soma = 0.0
    for i in range(A._linhas):
        for j in range(A._colunas):
            soma += A[i][j] * A[i][j]
    return sqrt(soma)


#Produto de Hadamard
def produto_hadamard(A: Matriz, B: Matriz) -> Matriz:
    if A.forma != B.forma:
        raise ValueError(f"Produto de Hadamard requer mesma forma: {A.forma} != {B.forma}.")
    resultado = [[A[i][j] * B[i][j] for j in range(A._colunas)] for i in range(A._linhas)]
    return Matriz(resultado)


#Transformações
#Aplicação de transformação
def aplicar_transformacao(A: Matriz, v: Vetor) -> Vetor:
    if A._colunas != v.dimensao:
        raise ValueError(f"Incompatível: matriz {A.forma} não pode atuar sobre vetor de dimensão {v.dimensao}.")
    resultado = []
    for i in range(A._linhas):
        soma = 0.0
        for j in range(A._colunas):
            soma += A[i][j] * v[j]
        resultado.append(soma)
    return Vetor(resultado)


#Verificação se uma trasformação é linear
def eh_transformacao_linear(transformacao, dimensao_dominio: int, n_amostras: int = 5) -> bool:
    escalares_teste = [0.0, 1.0, -1.0, 2.0, 0.5, -3.0]

    for k in range(n_amostras):
        u = Vetor([float((i + k + 1) % 7 - 3) for i in range(dimensao_dominio)])
        v = Vetor([float((i * 2 + k + 1) % 5 - 2) for i in range(dimensao_dominio)])

        #Verificação: T(u+v) = T(u) + T(v)
        Tu = transformacao(u)
        Tv = transformacao(v)
        T_soma = transformacao(u + v)
        soma_T = Tu + Tv

        if not vetores_aproximadamente_iguais(T_soma, soma_T):
            return False

        #Verificação: T(c·u) = c·T(u)
        for c in escalares_teste:
            T_escalado = transformacao(u * c)
            escalado_T = Tu * c
            if not vetores_aproximadamente_iguais(T_escalado, escalado_T):
                return False

    return True


#Comparação entre vetores
def vetores_aproximadamente_iguais(u: Vetor, v: Vetor) -> bool:
    if u.dimensao != v.dimensao:
        return False
    for i in range(u.dimensao):
        if valor_absoluto(u[i] - v[i]) > tolerancia_de_erro * 10:
            return False
    return True


#Matriz associada a transformação linear
def matriz_associada_transformacao(transformacao, dimensao_dominio: int) -> Matriz:
    colunas = []
    for j in range(dimensao_dominio):
        e_j = Vetor([1.0 if i == j else 0.0 for i in range(dimensao_dominio)])
        colunas.append(transformacao(e_j))

    dimensao_imagem = colunas[0].dimensao
    dados = [[colunas[j][i] for j in range(dimensao_dominio)] for i in range(dimensao_imagem)]
    return Matriz(dados)


#Base para núcleo
def base_nucleo(A: Matriz) -> list:
    R = gauss_jordan(A)
    m, n = A._linhas, A._colunas

    colunas_pivo = []
    linha_atual = 0
    for colunas in range(n):
        if linha_atual < m and valor_absoluto(R[linha_atual][colunas] - 1.0) < tolerancia_de_erro:
            eh_pivo = True
            for outra_linha in range(m):
                if outra_linha != linha_atual and valor_absoluto(R[outra_linha][colunas]) > tolerancia_de_erro:
                    eh_pivo = False
                    break
            if eh_pivo:
                colunas_pivo.append(colunas)
                linha_atual += 1

    colunas_livres = [c for c in range(n) if c not in colunas_pivo]
    base = []

    for coluna_livre in colunas_livres:
        vetor_base = [0.0] * n
        vetor_base[coluna_livre] = 1.0
        for i, col_pivo in enumerate(colunas_pivo):
            vetor_base[col_pivo] = -R[i][coluna_livre]
        base.append(Vetor(vetor_base))

    return base


#Dimensão da imagem
def imagem_dimensao(A: Matriz) -> int:
    return posto(A)

Overwriting algebra_linear.py
